> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# **IS 4487 LAB 9: CLUSTERING**

In this lab, we return to the **SF Rent** dataset that we used in **Lab 4: Data Understanding** and **Lab 5: Exploratory Data Analysis (EDA)**.

This time, we’ll explore how to segment the counties using both:
- **Manual clustering** based on business rules
- **Automatic clustering** using KMeans clustering

Segmentation helps identify meaningful groups within data, such as counties with high rent burden or low affordability. This is valuable for making targeted decisions in housing policy, urban planning, and social support.


## Outline

- Load and inspect the SF Rents dataset  
- Engineer and prepare features  
- Create manual segments using binning  
- Perform KMeans clustering for automatic segments  
- Visualize and compare results  

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Labs/lab_09_clustering.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## **Context: Rental Housing Market in the Bay Area**

## Dataset Overview
The data contains posts about Bay Area rental housing on Craigslist.

**Dataset:** `rent.csv`  
Source: [TidyTuesday 2022-07-05](https://github.com/rfordatascience/tidytuesday/blob/main/data/2022/2022-07-05/rent.csv)

| Variable       | Type       | Description |
|----------------|------------|-------------|
| `post_id`      | Categorical| Unique listing ID |
| `date`         | Numeric    | Listing date (numeric format) |
| `year`         | Integer    | Year of listing |
| `nhood`        | Categorical| Neighborhood |
| `city`         | Categorical| City |
| `county`       | Categorical| County |
| `price`        | Numeric    | Listing price (USD) |
| `beds`         | Numeric    | Number of bedrooms |
| `baths`        | Numeric    | Number of bathrooms |
| `sqft`         | Numeric    | Square footage |
| `room_in_apt`  | Binary     | 1 = room in apartment |
| `address`      | Categorical| Street address |
| `lat`          | Numeric    | Latitude |
| `lon`          | Numeric    | Longitude |
| `title`        | Text       | Listing title |
| `descr`        | Text       | Listing description |
| `details`      | Text       | Additional details |


## **Part 1: Importing the Data + Prepare for Segmentation**

### Instructions:
- Import necessary libraries.
- Import data from the rent.csv into a dataframe from the tidytuesday link.
- Use `.info()` and `.head()` to inspect the structure and preview the data.e structure and preview the data.
- Perform basic data cleanup including
  - Remove duplicates
  - Handle missing values (Optionally impute or filter variables)
  - Remove outliers (for price, beds, baths, sqft)
  - Fix data types

Note: we will do a few quick and easy steps for this Lab. Of course, in general, you will want to perform a more thorough cleaning .

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2022/2022-07-05/rent.csv'
df = pd.read_csv(url)

# Get a quick overview
print(df.info())

In [ ]:
display(df.head())

display(df.describe())

In [ ]:
# STEP 1: Drop duplicates with the same post ID
df = df.drop_duplicates(subset='post_id')

# STEP 2: Drop rows with nulls in essential columns - the ones we decide to use for clustering
essential = ['price', 'beds', 'baths', 'sqft', 'lat', 'lon']
df = df.dropna(subset=essential)

# STEP 3: Remove outliers (common-sense filtering)
df = df[df['price'].between(500, 20000)]
df = df[df['beds'].between(0, 10)]
df = df[df['baths'].between(0.5, 10)]
df = df[df['sqft'].between(100, 5000)]

# STEP 4: Convert data types if needed
df['beds'] = df['beds'].astype(int)
df['baths'] = df['baths'].astype(float)  # decimal values allowed
df['sqft'] = df['sqft'].astype(int)
df['price'] = df['price'].astype(int)

# STEP 5: Reset index
# creates an int index of 0,1,2,3...and moves the existing index to a column
# this is helpful if we want the index to actually be one of the columns. ISnce our index is 0,1,2, already, we will just drop it.
df = df.reset_index(drop=True)

# Preview cleaned data
df.info()

In [ ]:
df.sample(n=5)

While this drops a significant amount of data, we wouldnot have been able to do clustering on those observations since they had missing values. Alternatively, if we had enough information/intuition, we could have imputed some of the missing values to keep more of the rows.

## **Part 2: Engineer and Prepare Features**

We’ll select features for clustering:
- Property: `price`, `beds`, `baths`, `sqft`
- Geographic: `lat`, `lon`

We need to ensure that all these features that we use for clustering are in the same scale. We can either choose standardization (mean = 0, var = 1) or minmax normalization (0-1). We’ll standardize features to ensure fair weighting in distance-based clustering.

### Why This Matters:
Scaling avoids giving larger-scale variables (like `price`) more influence in ML algorithms, such as Clustering, that are based on distance measures. Otherwise we will get clitsre that are heavily influenced by only the larger scale features, ignoring the smaller scaled features, which need not reflect reality.

### Things to think about:
- Should all variables be scaled?
- Do geographic coordinates need scaling?


In [ ]:
from sklearn.preprocessing import StandardScaler

# Select and scale features
features = ['price', 'beds', 'baths', 'sqft', 'lat', 'lon']
segment_df = df[features].copy()

# apply standardization
scaler = StandardScaler()
scaled_data = scaler.fit_transform(segment_df)

# Convert back to DataFrame
scaled_df = pd.DataFrame(scaled_data, columns=features)
scaled_df.describe()


### 🔧 **Try It Yourself - Part 2**

2.1. Why should lat/lon (latitude and longitude) be scaled before clustering? What would happen if they weren’t? (There’s no coding here, just write your response below.)


🔧 2.1 Add comment here:

## **Part 3: Create Manual Segments Using Binning**

Let’s group listings by price into:
  - **Low**: < $2,000

  - **Mid**: \$2,000 - $4,000
  
  - **High**: > $4,000

### Why This Matters:
Manual bins based on thresholds offer simple segmentation — useful when business rules exist.

### Things to think about:
- Why might one want to create such manual bins or segments based on one or few features?
- Are fixed or custom cutoffs (that use `pd.cut()`) better than percentiles/quantiles using `pd.qcut()`?
- Should you also bin square footage?


In [ ]:
# Create price segments
df['price_segment'] = pd.cut(
    df['price'],
    bins=[0, 2000, 4000, float('inf')],
    labels=['Low', 'Mid', 'High']
)

df['price_segment'].value_counts()


### 🔧 **Try It Yourself - Part 3**

3.1. Create a column called `sqft_segment` using bins:  
  - Small: `< 800`,
  - Medium: `800–1400`,
  - Large: `>1400`  

3.2. Count how many listings fall into each `sqft_segment` using `.value_counts()`  

3.3. Use `.head()` or `.sample()` to preview 5 values of both `price_segment` and `sqft_segment`

In [ ]:
# 🔧 3.1. Add code here


# 🔧 3.2. Add code here


# 🔧 3.3. Add code here



## **Part 4: Perform K-Means Clustering**

We’ll create two sets of clusters:
1. **Home feature-based** (price, beds, baths, sqft)
2. **Geographic-based** (lat, lon)

### Why This Matters:
Unsupervised clustering finds hidden patterns — useful for market segmentation, targeting, etc.

### Things to think about:
- How many clusters should you use?
- How do results differ between property and location clusters?


In [ ]:
from sklearn.cluster import KMeans

# Select only the standardized property/home features
X_home = scaled_df[['price', 'beds', 'baths', 'sqft']]

# Apply KMeans clustering with 4 clusters
kmeans_home = KMeans(n_clusters=4, random_state=35)     # set the seed as any random int
scaled_df['homefeature_cluster'] = kmeans_home.fit_predict(X_home)

# Show number of listings in each cluster
scaled_df['homefeature_cluster'].value_counts()


In [ ]:
# Select only standardized geographic coordinate features
X_geo = scaled_df[['lat', 'lon']]

# Apply KMeans clustering with 5 clusters
kmeans_geo = KMeans(n_clusters=5, random_state=35)
scaled_df['geo_cluster'] = kmeans_geo.fit_predict(X_geo)

# Show number of listings in each geographic cluster
scaled_df['geo_cluster'].value_counts()

### 🔧 **Try It Yourself - Part 4**

4.1. Run KMeans again using `k=3` (3 clusters instead of 4) for Cluster based on home_features. Save these clusters as a new column in scaled_df, say, `homefeature_cluster2`. Then compare the value counts of the old K-Means cluster with your new K-Means cluster using `pd.crosstab()`.

4.2. Plot a histogram of `price` grouped by `homefeature_cluster2`. Add appropriate titles, heading ad axis labels.

In [ ]:
# 🔧 4.1. Add code here



# 🔧 4.2. Add code here


## **Part 5: Visualize and Compare Results**

We’ll now visualize:
- Clusters on a map (lat/lon)
- Clusters in price vs sqft space

### Why This Matters:
Visual validation helps determine if clusters are interpretable and useful.

### Things to think about:
- Are location clusters geographically meaningful?
- Do property clusters separate by price or size?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot latitude vs longitude colored by geographic cluster
plt.figure(figsize=(10,6))
sns.scatterplot(data=scaled_df,
                x='lon',
                y='lat',
                hue='geo_cluster',
                palette='tab10')

plt.title("K-Means Geographic Clusters (lat/lon)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title='Geo Cluster')
plt.show()

In [ ]:
# Plot price vs square footage colored by feature cluster
plt.figure(figsize=(10,6))
sns.scatterplot(data=scaled_df,
                x='price',
                y='sqft',
                hue='feature_cluster',
                palette='Set2')

plt.title("K-Means Property Clusters (Price vs Sqft)")
plt.xlabel("Price")
plt.ylabel("Square Footage")
plt.legend(title='Feature Cluster')
plt.show()

### 🔧 **Try It Yourself - Part 5**

In past assignments you’ve created scatterplots where the size of each point depended on how large each value was. And you’ve created scatterplots where the color of each point was assigned many or few colors.



5.1. Re-plot the above `price` vs `sqft` scatterplot with the following: Add `beds` as the **point size** and add `baths` as the **point style**

5.2. Group by `feature_cluster` and calculate:
  - Average `price`
  - Average `sqft`
  - Average `beds`  


In [ ]:
# 🔧 5.1. Add code here to replot the scatterplot

# 🔧 5.2. Group by feature_cluster and calculate average price, sqft, and beds


## **Part 6: Evaluate the Fit for your Clusters**

We’ll now use three methods to evaluate the fit:
- **WCSS**: Within-cluster sums of squares
- **Silhouette score**
- **Davies-bouldin index**

### Why This Matters:
Clusters are difficult to visualize when they are based on more than 3 variables.  A statistical score will evaluate the fit across all of the variables.

### Things to think about:
- Lower WCSS = tighter, better-defined clusters
- Silhouette score ranges from -1 to 1.  Higher values = better clustering
- Lower Davies-Boulding Index = better clustering



### 🔧 **Try It Yourself – Part 6**

Calculate the three scores for your **feature-based** cluster model by using the following methods:

6.1. WCSS

6.2. Silhouette score

6.3. Davies-bouldin index

Typically, we would compare these scores for different k to help us pick the optimal number of clusters.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# 🔧 6.1. Add code here - WCSS
wcss = kmeans_home.inertia_
print(f"WCSS: {wcss}")

# 🔧 6.2. Add code here - Silhouette score
silhouette_avg = silhouette_score(X_home, scaled_df['homefeature_cluster'])
print(f"Silhouette Score: {silhouette_avg}")

# 🔧 6.3. Add code here - Davies-bouldin index
db_index = davies_bouldin_score(X_home, scaled_df['homefeature_cluster'])
print(f"Davies-Bouldin Index: {db_index}")

## 🔧 **Part 7: Reflection (100 words or less per question)**

7.1. Which method—manual binning or KMeans clustering—gave you more useful insights? Why?

7.2. How might missing data or outliers affect your segmentation results?


🔧 7.1. Add comment here


🔧 7.2. Add comment here



## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_09_LastnameFirstname.ipynb"